In [1]:
!pip install pybamm

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from run_code import run_battery_simulation

# Example usage
a_nmc, b_nmc, c_nmc, d_nmc = 153.1353114622245, -299.5867812542784, 182.0307751506729, -65.20873735054764 #uniform for overall diffusion if fine, but what about individual a, b, c, d; gaussian for the overall diffusion  (biggist SD informed by the literature)
my_graphite_diff_parameter = 1 #gaussian is good for figuring out the diffusion - but what if mean if 4 SDs away - is a uniform prior betteR? Could just do a big SD. (SD: 10^-3? Something big that is informed by literature) 
sep_por, neg_por, pos_por = 0.4, 0.25, 0.335 #uniform for porosity is fine; between 20 and 45 percent for porosities; 5 percent error (do we even need to ); 
cap_dl_neg = 0.2 #Will depend on particle size, particle size distributiosn are Gaussian; (SD: averasge value about 10 or 15 microns - a SD might be 5 microns for particle sizes themselves) (so 0.2 mean and 0.1 SD) 

#lectrode capacity and stoichiometry variables? Tune the stoichiometric operating window 
#to figure out the initial state of charge 


#don't go above 1.5C current  - could even do 2 C

p1 = 0.5  # fraction of C-rate - 0.1 to 2 limit uniform - initial discharge amplitude
p2 = 598.6 # discharge duration - seconds - make sure this is duration of current discharge  - specify as a function of p1 to be between 2% and 10%SOC - change more than 2% of the SOC
p3 = 1800/2  # rest after discharge -  30 minutes in seconds - max 30 minutes, minimum 100 seconds
p4 = 1.0 # HPPC amplitude - fraction of C-rate - HPPC - 0.1 to 2 limit uniform
p5 = 10.0 # HPPC duration - seconds - 1 to 20 seconds 
p6 = 600  # Rest between HPPCs- 10 minutes in seconds - min 10 seconds max 600 seconds 

p7 = 10e-3 #start_frequency - maybe uniform between 1e-3 to 10e-3?
p8 = 3000 #end_time - quarter of a period of p7 to full period of p7

# Run the simulation
time_data, terminal_voltage_data = run_battery_simulation(a_nmc, b_nmc, c_nmc, d_nmc, my_graphite_diff_parameter, sep_por, neg_por, pos_por, cap_dl_neg, p1, p2, p3, p4, p5, p6, p7, p8)


In [1]:
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from run_code import run_battery_simulation
import os

def single_run(params):
    # Unpack your parameters here
    a_nmc, b_nmc, c_nmc, d_nmc, diff_param, sep_por, neg_por, pos_por, cap_dl_neg, p1, p2, p3, p4, p5, p6, p7, p8 = params
    
    # Run the simulation with the given parameters
    time_data, voltage_data = run_battery_simulation(a_nmc, b_nmc, c_nmc, d_nmc, diff_param, sep_por, neg_por, pos_por, cap_dl_neg, p1, p2, p3, p4, p5, p6, p7, p8)
    
    return time_data, voltage_data

def generate_parameters():
    from scipy.stats import multivariate_normal, norm
    import numpy as np
    # a_nmc, b_nmc, c_nmc, d_nmc joint posterior from inference
    sigma = np.array([[ 414.16743966, -439.53022219,  157.53082159,  -36.59908635], \
           [-439.53022219,  475.21734538, -170.37237657,   41.13810463], \
           [ 157.53082159, -170.37237657,   61.12237867,  -14.75655731], \
           [ -36.59908635,   41.13810463,  -14.75655731,    3.8372772 ]])
    mu = np.array([-2.29714210e+01, -1.23599647e-02, -1.09287243e+00,  1.62538939e+00])
    a_nmc, b_nmc, c_nmc, d_nmc = multivariate_normal.rvs(mu,sigma,size=1)
    
    # graphite_diff_parameter posterior from inference
    stdev = np.sqrt(3.62066527e-05)
    mu = 1.
    graphite_diff_parameter = norm.rvs(mu, stdev, size=1)[0]
    
    # sep_por, neg_por, pos_por
    lb = 0.2
    ub = 0.45
    sep_por = np.random.uniform(lb,ub)
    neg_por = np.random.uniform(lb,ub)
    pos_por = np.random.uniform(lb,ub)
    
    # cap_dl_neg
    mu = 0.2
    stdev = 0.05
    cap_dl_neg = norm.rvs(mu, stdev, size=1)[0]
    
    # p1
    lb = 0.1
    ub=2.
    p1 = np.random.uniform(lb,ub)
    
    # p2 
    lb = 72./p1
    ub = 360./p1
    p2 = np.random.uniform(lb,ub)
    
    # p3
    lb = 100.
    ub = 1800.
    p3 = np.random.uniform(lb,ub)
    
    # p4
    lb = 0.1
    ub = 2
    p4 = np.random.uniform(lb,ub)
    
    # p5
    lb = 1.
    ub = 20
    p5 = np.random.uniform(lb,ub)
    
    # p6
    lb = 10.
    ub = 600.
    p6 = np.random.uniform(lb,ub)
    
    # p7
    lb = 0.001
    ub = 0.01
    p7 = np.random.uniform(lb,ub)
    
    # p8
    lb = 0.25/p7
    ub = 1/p7
    p8 = np.random.uniform(lb,ub)
    
    #returns a_nmc, b_nmc, c_nmc, d_nmc, graphite_diff_parameter, sep_por, neg_por, pos_por, cap_dl_neg, p1, p2, p3, p4, p5, p6, p7, p8
    return a_nmc, b_nmc, c_nmc, d_nmc, graphite_diff_parameter, sep_por, neg_por, pos_por, cap_dl_neg, p1, p2, p3, p4, p5, p6, p7, p8

def save_results(time_data, voltage_data, run_id):
    # Define your directory for saving results
    results_dir = "simulation_results"
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)

    # Save the time data and voltage data as .npy files
    np.save(os.path.join(results_dir, f"time_data_{run_id}.npy"), time_data)
    np.save(os.path.join(results_dir, f"voltage_data_{run_id}.npy"), voltage_data)

if __name__ == "__main__":
    parameter_sets = [generate_parameters() for _ in range(100)]
    
    dynamic_number_of_workers = os.cpu_count() - 1 #can remove -1, just here to keep one core free
    with ProcessPoolExecutor(max_workers=dynamic_number_of_workers) as executor:
        for i, (time_data, voltage_data) in enumerate(executor.map(single_run, parameter_sets)):
            save_results(time_data, voltage_data, i)
            print(f"Run {i} completed and saved.")

            
"""
DEsign parameter definitions -> 
p1 = 0.5  # fraction of C-rate - 0.1 to 2 limit uniform - initial discharge amplitude
p2 = 598.6 # discharge duration - seconds - make sure this is duration of current discharge  - specify as a function of p1 to be between 2% and 10%SOC - change more than 2% of the SOC
p3 = 1800/2  # rest after discharge -  30 minutes in seconds - max 30 minutes, minimum 100 seconds
p4 = 1.0 # HPPC amplitude - fraction of C-rate - HPPC - 0.1 to 2 limit uniform
p5 = 10.0 # HPPC duration - seconds - 1 to 20 seconds 
p6 = 600  # Rest between HPPCs- 10 minutes in seconds - min 10 seconds max 600 seconds 

p7 = 10e-3 #start_frequency - maybe uniform between 1e-3 to 10e-3?
p8 = 3000 #end_time - quarter of a period of p7 to full period of p7
"""

Default Solver: CasADi solver with 'safe' mode
Default Solver: CasADi solver with 'safe' mode
Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:          CasADi solver with 'safe' mode  CasADi solver with 'safe' modeCasADi solver with 'safe' modeCasADi solver with 'safe' modeCasADi solver with 'safe' modeCasADi solver with 'safe' modeCasADi solver with 'safe' modeCasADi solver with 'safe' modeCasADi solver with 'safe' modeCasADi solver with 'safe' mode
CasADi solver with 'safe' modeCasADi solver with 'safe' mode


  







CasADi solver with 'safe' modeDefault Solver:Default Solver:Default Solver:CasADi solver with 'safe' modeDefault Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:Default Solver:          CasADi solver with 'safe' modeCasADi solver with 'safe' mode

2024-03-28 13:15:24.340 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment
2024-03-28 13:15:25.404 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment
2024-03-28 13:15:25.410 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment
2024-03-28 13:15:25.519 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment
2024-03-28 13:15:25.562 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment
2024-03-28 13:15:25.612 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment
2024-03-28 13:15:25.657 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment
2024-03-28 13:15:35.332 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.